In [2]:
#!/usr/bin/env python3
import requests
import json

def fetch_swagger_metadata(url, output_file):
    try:
        response = requests.get(url)
        response.raise_for_status()
        # Parse JSON response
        swagger_metadata = response.json()
        # Write metadata to file
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(swagger_metadata, f, indent=2)
        print(f"Swagger metadata successfully saved to '{output_file}'.")
    except requests.exceptions.RequestException as e:
        print(f"An error occurred while fetching the Swagger metadata: {e}")

swagger_url = "https://api.ed-fi.org/v7.2/api/swagger.json"
output_filename = "swagger_metadata.json"
    
fetch_swagger_metadata(swagger_url, output_filename)


Swagger metadata successfully saved to 'swagger_metadata.json'.


In [24]:
#parse the swagger metadata to get the school json schmma
def parse_swagger_metadata(swagger_metadata_file):
    with open(swagger_metadata_file, "r", encoding="utf-8") as f:
        swagger_metadata = json.load(f)
    # Get the schema for the School resource
    school_schema = swagger_metadata["components"]["schemas"].get("edFi_school")
    if school_schema is None:
        print("The 'School' schema is not found in the swagger metadata.")
    return school_schema

if __name__ == "__main__":
    swagger_metadata_file = "swagger_metadata.json"
    school_schema = parse_swagger_metadata(swagger_metadata_file)
    if school_schema:
        print(school_schema)

{'required': ['schoolId', 'nameOfInstitution', 'gradeLevels', 'educationOrganizationCategories'], 'type': 'object', 'properties': {'id': {'type': 'string', 'description': ''}, 'educationOrganizationCategories': {'type': 'array', 'items': {'$ref': '#/components/schemas/edFi_educationOrganizationCategory'}, 'description': 'An unordered collection of educationOrganizationCategories. The classification of the education agency within the geographic boundaries of a state according to the level of administrative and operational control granted by the state.'}, 'gradeLevels': {'type': 'array', 'items': {'$ref': '#/components/schemas/edFi_schoolGradeLevel'}, 'description': 'An unordered collection of schoolGradeLevels. The grade levels served at the school.'}, 'schoolId': {'type': 'integer', 'description': 'The identifier assigned to a school. It must be distinct from any other identifier assigned to educational organizations, such as a LocalEducationAgencyId, to prevent duplication.', 'format'

In [26]:
import pandas as pd
import json
import re

def parse_path(path):
    """
    Parse a dot-delimited path string into components.
    Each component is a tuple of (name, index) where index is an integer if the component is an array element.
    For example: "addresses[0].periods[1].beginDate" becomes:
      [("addresses", 0), ("periods", 1), ("beginDate", None)]
    """
    components = []
    # Split the path by dot and process each component
    for part in path.split('.'):
        match = re.match(r'([^\[]+)(?:\[(\d+)\])?', part)
        if match:
            name, index = match.groups()
            components.append((name, int(index) if index is not None else None))
    return components

def set_nested_value(obj, components, value):
    """
    Set a value in a nested dictionary/list structure based on a list of (name, index) components.
    If an index is provided, then the component is assumed to be an array.
    """
    current = obj
    for i, (name, index) in enumerate(components[:-1]):
        if index is not None:
            if name not in current:
                current[name] = []
            # Ensure the list is long enough
            while len(current[name]) <= index:
                current[name].append({})
            current = current[name][index]
        else:
            if name not in current:
                current[name] = {}
            current = current[name]
    
    # Set the value for the last component
    last_name, last_index = components[-1]
    if last_index is not None:
        if last_name not in current:
            current[last_name] = []
        while len(current[last_name]) <= last_index:
            current[last_name].append({})
        current[last_name][last_index] = value
    else:
        current[last_name] = value

def map_row_to_json(row):
    """
    Map a single CSV row to a nested JSON object.
    The CSV header paths (after stripping '/ed-fi/schools/') define the structure.
    For example, a header like:
      /ed-fi/schools/addresses[0].periods[0].beginDate
    will produce a nested structure where 'addresses' is an array of objects,
    and each address object has a 'periods' array of objects.
    """
    json_obj = {}
    for col in row.index:
        # Skip template headers containing [n]
        if pd.notna(row[col]) :
            # Remove the constant prefix if exists
            path = col.replace('/ed-fi/schools/', '')
            # Parse the path into components
            components = parse_path(path)
            set_nested_value(json_obj, components, row[col])
    return json_obj

# Read CSV and process rows dynamically based on header paths
csv_file_path = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.csv'
df = pd.read_csv(csv_file_path)
json_data = df.apply(map_row_to_json, axis=1).tolist()

# Write the generated JSON to a file
json_file_path = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.json'
with open(json_file_path, 'w') as f:
    json.dump(json_data, f, indent=2)

print(f"JSON data saved to {json_file_path}")

JSON data saved to /home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.json


In [3]:
import pandas as pd
import json
import re

def parse_path(path):
    """
    Parse a dot-delimited path string into components.
    Each component is a tuple of (name, index) where index is an integer if the component is an array element.
    For example: "addresses[0].periods[1].beginDate" becomes:
      [("addresses", 0), ("periods", 1), ("beginDate", None)]
    """
    components = []
    for part in path.split('.'):
        match = re.match(r'([^\[]+)(?:\[(\d+)\])?', part)
        if match:
            name, index = match.groups()
            components.append((name, int(index) if index is not None else None))
    return components

def recursive_set(obj, comps, value):
    """
    Recursively set the 'value' in the nested structure 'obj' using the list of components.
    Each component is a tuple (key, index). If index is provided, the key represents a list.
    """
    if not comps:
        return

    key, index = comps[0]

    # Final component: set the value
    if len(comps) == 1:
        if index is not None:
            if key not in obj:
                obj[key] = []
            while len(obj[key]) <= index:
                obj[key].append({})
            obj[key][index] = value
        else:
            obj[key] = value
        return

    # Not final: ensure the key exists and is of correct type (dict or list)
    if index is not None:
        if key not in obj:
            obj[key] = []
        # Expand the list if needed
        while len(obj[key]) <= index:
            obj[key].append({})
        # Recurse into the proper list element
        recursive_set(obj[key][index], comps[1:], value)
    else:
        if key not in obj:
            obj[key] = {}
        recursive_set(obj[key], comps[1:], value)

def set_nested_value(obj, components, value):
    """
    Wrapper to set the nested value in obj using recursive_set.
    """
    recursive_set(obj, components, value)

def map_row_to_json(row):
    """
    Map a single CSV row to a nested JSON object.
    The CSV header paths (after stripping '/ed-fi/schools/') define the structure.
    For example, a header like:
      /ed-fi/schools/addresses[0].periods[0].beginDate
    will produce a nested structure where 'addresses' is an array of objects,
    and each address object has a 'periods' array of objects.
    """
    json_obj = {}
    for col in row.index:
        # Process only columns with actual data and skip template headers (with [n])
        if pd.notna(row[col]) :
            # Remove the prefix /.*/.*/ from the column name using regex
            
            

            components = parse_path(path)
            set_nested_value(json_obj, components, row[col])
    return json_obj

# Read CSV and build JSON objects dynamically based on header paths
csv_file_path = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.csv'
df = pd.read_csv(csv_file_path)
json_data = df.apply(map_row_to_json, axis=1).tolist()

# Write JSON output to file
json_file_path = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.json'
with open(json_file_path, 'w') as f:
    json.dump(json_data, f, indent=2)

print(f"JSON data saved to {json_file_path}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.csv'

In [20]:
import os
import requests

# Retrieve credentials and base URL from environment variables
client_id = os.getenv('EDFI_API_CLIENT_ID')
client_secret = os.getenv('EDFI_API_CLIENT_SECRET')
base_url = 'https://api.ed-fi.org:443/v7.2/api/data/v3'
def get_access_token(client_id, client_secret):
    url = "https://api.ed-fi.org/v7.2/api/oauth/token"
    data = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret
    }
    response = requests.post(url, data=data)
    response.raise_for_status()
    return response.json().get('access_token')


if not client_id or not client_secret :
    raise EnvironmentError("Missing one or more environment variables: EDFI_API_CLIENT_ID, EDFI_API_CLIENT_SECRET")
# get authenticaion token


# Construct the API endpoint URL
endpoint = "/ed-fi/schools?offset=0&limit=20"
url = f"{base_url}{endpoint}"
token = get_access_token(client_id, client_secret)
headers = {
    'Authorization': f'Bearer {token}',
    'Accept': 'application/json',
    'Content-Type': 'application/json',

}
#make a get request to the api endpoint
response = requests.get(url, headers=headers)

if response.ok:
    print(response.json())
else:
    response.raise_for_status()

[{'id': '04eaa4df3f384e46bd810730c8a9c957', 'schoolId': 5, 'nameOfInstitution': 'UT Austin College of Education Graduate', 'addresses': [{'addressTypeDescriptor': 'uri://ed-fi.org/AddressTypeDescriptor#Physical', 'city': 'Austin', 'postalCode': '78712', 'stateAbbreviationDescriptor': 'uri://ed-fi.org/StateAbbreviationDescriptor#TX', 'streetNumberName': '1912 Speedway Stop D5000', 'nameOfCounty': 'Travis', 'periods': []}], 'educationOrganizationCategories': [{'educationOrganizationCategoryDescriptor': 'uri://tpdm.ed-fi.org/EducationOrganizationCategoryDescriptor#Educator Preparation Provider'}], 'identificationCodes': [], 'indicators': [], 'institutionTelephones': [], 'internationalAddresses': [], '_ext': {'tpdm': {'postSecondaryInstitutionReference': {'postSecondaryInstitutionId': 6000203, 'link': {'rel': 'PostSecondaryInstitution', 'href': '/ed-fi/postSecondaryInstitutions/8a9ce45770534a7fbf739898062cd7c1'}}}}, 'schoolCategories': [], 'gradeLevels': [{'gradeLevelDescriptor': 'uri://ed

In [30]:
import pandas as pd
import json
import re

def parse_path(path):
    """
    Parse a dot-delimited path string into components.
    Each component is a tuple of (name, index) where index is an integer if the component is an array element.
    For example: "addresses[0].periods[1].beginDate" becomes:
      [("addresses", 0), ("periods", 1), ("beginDate", None)]
    """
    components = []
    for part in path.split('.'):
        match = re.match(r'([^\[]+)(?:\[(\d+)\])?', part)
        if match:
            name, index = match.groups()
            components.append((name, int(index) if index is not None else None))
    return components

def recursive_set(obj, comps, value):
    """
    Recursively set the 'value' in the nested structure 'obj' using the list of components.
    Each component is a tuple (key, index). If index is provided, the key represents a list.
    """
    if not comps:
        return

    key, index = comps[0]

    # Final component: set the value
    if len(comps) == 1:
        if index is not None:
            if key not in obj:
                obj[key] = []
            while len(obj[key]) <= index:
                obj[key].append({})
            obj[key][index] = value
        else:
            obj[key] = value
        return

    # Not final: ensure the key exists and is of correct type (dict or list)
    if index is not None:
        if key not in obj:
            obj[key] = []
        while len(obj[key]) <= index:
            obj[key].append({})
        recursive_set(obj[key][index], comps[1:], value)
    else:
        if key not in obj:
            obj[key] = {}
        recursive_set(obj[key], comps[1:], value)

def set_nested_value(obj, components, value):
    recursive_set(obj, components, value)

def map_row_to_json(row):
    """
    Map a single CSV row to a nested JSON object.
    The CSV header paths (after stripping '/ed-fi/schools/') define the structure.
    For example, a header like:
      /ed-fi/schools/addresses[0].periods[0].beginDate
    will produce a nested structure where 'addresses' is an array of objects,
    and each address object has a 'periods' array of objects.
    """
    json_obj = {}
    for col in row.index:
        if pd.notna(row[col]) and '[n]' not in col:
            path = col.replace('/ed-fi/schools/', '')
            components = parse_path(path)
            set_nested_value(json_obj, components, row[col])
    return json_obj

# Read CSV and build JSON objects dynamically based on header paths
csv_file_path = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.csv'
df = pd.read_csv(csv_file_path)
json_data = df.apply(map_row_to_json, axis=1).tolist()

# Write the JSONL file: each line is a JSON object
jsonl_file_path = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.jsonl'
with open(jsonl_file_path, 'w') as f:
    for row_obj in json_data:
        f.write(json.dumps(row_obj) + "\n")

print(f"JSONL data saved to {jsonl_file_path}")

JSONL data saved to /home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.jsonl


In [ ]:
import pandas as pd
import json
import re

DEFAULT_NAMESPACE = "uri://ed-fi.org"

def parse_path(path):
    """
    Parse a dot-delimited path string into components.
    Each component is a tuple of (name, index) where index is an integer if the component is an array element.
    For example: "addresses[0].periods[1].beginDate" becomes:
      [("addresses", 0), ("periods", 1), ("beginDate", None)]
    """
    components = []
    for part in path.split('.'):
        match = re.match(r'([^\[]+)(?:\[(\d+)\])?', part)
        if match:
            name, index = match.groups()
            components.append((name, int(index) if index is not None else None))
    return components

def recursive_set(obj, comps, value):
    """
    Recursively set the 'value' in the nested structure 'obj' using the list of components.
    Each component is a tuple (key, index). If index is provided, the key represents a list.
    """
    if not comps:
        return

    key, index = comps[0]

    # Final component: set the value
    if len(comps) == 1:
        if index is not None:
            if key not in obj:
                obj[key] = []
            while len(obj[key]) <= index:
                obj[key].append({})
            obj[key][index] = value
        else:
            obj[key] = value
        return

    # Not final: ensure the key exists and is of correct type (dict or list)
    if index is not None:
        if key not in obj:
            obj[key] = []
        while len(obj[key]) <= index:
            obj[key].append({})
        recursive_set(obj[key][index], comps[1:], value)
    else:
        if key not in obj:
            obj[key] = {}
        recursive_set(obj[key], comps[1:], value)

def set_nested_value(obj, components, value):
    recursive_set(obj, components, value)

def update_descriptors(obj, namespace=DEFAULT_NAMESPACE):
    """
    Recursively traverse the JSON object and update any field whose key ends with "Descriptor".
    The new value is formatted as:
        {namespace}/{key}#{original_value}
    For example, if key is "AbsenceEventCategoryDescriptor" and value is "Compensatory leave time",
    it becomes "uri://ed-fi.org/AbsenceEventCategoryDescriptor#Compensatory leave time".
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, (dict, list)):
                update_descriptors(v, namespace)
            else:
                if k.endswith("Descriptor") and isinstance(v, str) and "#" not in v:
                    obj[k] = f"{namespace}/{k}#{v}"
    elif isinstance(obj, list):
        for item in obj:
            update_descriptors(item, namespace)

def map_row_to_json(row):
    """
    Map a single CSV row to a nested JSON object.
    The CSV header paths (after stripping '/ed-fi/schools/') define the structure.
    For example, a header like:
      /ed-fi/schools/addresses[0].periods[0].beginDate
    will produce a nested structure where 'addresses' is an array of objects,
    and each address object has a 'periods' array of objects.
    
    The "SchoolYear" column is ignored.
    """
    json_obj = {}
    for col in row.index:
        if col == "SchoolYear":  # ignore the school year column
            continue
        if pd.notna(row[col]):
            path = re.sub(r'^/[^/]+/[^/]+/', '', col)
            components = parse_path(path)
            set_nested_value(json_obj, components, row[col])
    # Update descriptors in the resulting JSON object
    update_descriptors(json_obj, DEFAULT_NAMESPACE)
    return json_obj


csv_file_path = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.csv'
df = pd.read_csv(csv_file_path)
json_data = df.apply(map_row_to_json, axis=1).tolist()


jsonl_file_path = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.jsonl'
with open(jsonl_file_path, 'w') as f:
    for row_obj in json_data:
        f.write(json.dumps(row_obj) + "\n")

print(f"JSONL data with updated descriptors (ignoring SchoolYear) saved to {jsonl_file_path}")

JSONL data with updated descriptors (ignoring SchoolYear) saved to /home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.jsonl


In [ ]:
import pandas as pd
import json
import re

DEFAULT_NAMESPACE = "uri://ed-fi.org"

# Load descriptors CSV once and build a lookup table.
# The CSV file is assumed to have these columns:
# DESCRIPTOR_NAME,Owner,NAMESPACE,CODE_VALUE,SHORT_DESCRIPTION,DESCRIPTION
def load_descriptor_lookup(csv_path):
    df = pd.read_csv(csv_path)
    lookup = {}
    # Build a key: (descriptor_type, code_value_lower) -> "NAMESPACE#CODE_VALUE"
    for _, row in df.iterrows():
        # The CSV uses a descriptor name like "absence_event_category_descriptors".
        # We'll normalize our descriptor field from the JSON to a similar key.
        descriptor_name = str(row['DESCRIPTOR_NAME']).strip()
        code_val = str(row['CODE_VALUE']).strip()
        namespace = str(row['NAMESPACE']).strip()
        # key is a tuple: (descriptor_name, code_value_lower)
        lookup[(descriptor_name, code_val.lower())] = f"{namespace}#{code_val}"
    return lookup

# Convert a JSON descriptor field name to the CSV descriptor name format.
# For example, a JSON key "AbsenceEventCategoryDescriptor" becomes "absence_event_category_descriptors"
def normalize_descriptor_field(json_field):
    # Insert underscore before uppercase letters (except first char), then lowercase it.
    normalized = re.sub(r'(?<!^)(?=[A-Z])', '_', json_field).lower()
    # In our CSV, descriptor names are plural; add an "s" if not already present.
    if not normalized.endswith("s"):
        normalized += "s"
    return normalized

# Cache the lookup table once
DESCRIPTOR_LOOKUP = load_descriptor_lookup('/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/Descriptors.csv')

def parse_path(path):
    """
    Parse a dot-delimited path string into components.
    Each component is a tuple of (name, index) where index is an integer if the component is an array element.
    For example: "addresses[0].periods[1].beginDate" becomes:
      [("addresses", 0), ("periods", 1), ("beginDate", None)]
    """
    components = []
    for part in path.split('.'):
        match = re.match(r'([^\[]+)(?:\[(\d+)\])?', part)
        if match:
            name, index = match.groups()
            components.append((name, int(index) if index is not None else None))
    return components

def recursive_set(obj, comps, value):
    """
    Recursively set the 'value' in the nested structure 'obj' using the list of components.
    Each component is a tuple (key, index). If index is provided, the key represents a list.
    """
    if not comps:
        return

    key, index = comps[0]

    # Final component: set the value
    if len(comps) == 1:
        if index is not None:
            if key not in obj:
                obj[key] = []
            while len(obj[key]) <= index:
                obj[key].append({})
            obj[key][index] = value
        else:
            obj[key] = value
        return

    # Not final: ensure the key exists and is of correct type (dict or list)
    if index is not None:
        if key not in obj:
            obj[key] = []
        while len(obj[key]) <= index:
            obj[key].append({})
        recursive_set(obj[key][index], comps[1:], value)
    else:
        if key not in obj:
            obj[key] = {}
        recursive_set(obj[key], comps[1:], value)

def set_nested_value(obj, components, value):
    recursive_set(obj, components, value)

def update_descriptors(obj, namespace=DEFAULT_NAMESPACE):
    """
    Recursively traverse the JSON object and update any field whose key ends with "Descriptor".
    Instead of a default formatting, this version looks up a cached value in the descriptor lookup.
    If the descriptor lookup finds a match for the (normalized field name, original value), the value is replaced.
    Otherwise, the value is replaced with: {namespace}/{key}#{original_value}.
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, (dict, list)):
                update_descriptors(v, namespace)
            else:
                if k.endswith("Descriptor") and isinstance(v, str):
                    # Normalize the JSON field name to match our CSV descriptor name
                    normalized_field = normalize_descriptor_field(k)
                    lookup_key = (normalized_field, v.strip().lower())
                    if lookup_key in DESCRIPTOR_LOOKUP:
                        obj[k] = DESCRIPTOR_LOOKUP[lookup_key]
                    else:
                        # Fallback if no matching descriptor is found:
                        obj[k] = f"{namespace}/{k}#{v}"
    elif isinstance(obj, list):
        for item in obj:
            update_descriptors(item, namespace)

def map_row_to_json(row):
    """
    Map a single CSV row to a nested JSON object.
    The CSV header paths (after stripping '/ed-fi/schools/') define the structure.
    For example, a header like:
      /ed-fi/schools/addresses[0].periods[0].beginDate
    will produce a nested structure where 'addresses' is an array of objects,
    and each address object has a 'periods' array of objects.
    
    The "SchoolYear" column is ignored.
    """
    json_obj = {}
    for col in row.index:
        if col == "SchoolYear":  # ignore the school year column
            continue
        if pd.notna(row[col]):
            path = col.replace('/ed-fi/schools/', '')
            components = parse_path(path)
            set_nested_value(json_obj, components, row[col])
    # Update descriptors in the resulting JSON object using our cached lookup
    update_descriptors(json_obj, DEFAULT_NAMESPACE)
    return json_obj

# Read CSV and build JSON objects dynamically based on header paths
csv_file_path = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.csv'
df = pd.read_csv(csv_file_path)
json_data = df.apply(map_row_to_json, axis=1).tolist()

# Write the JSONL file: each line is a JSON object
jsonl_file_path = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.jsonl'
with open(jsonl_file_path, 'w') as f:
    for row_obj in json_data:
        f.write(json.dumps(row_obj) + "\n")

print(f"JSONL data with updated descriptors saved to {jsonl_file_path}")

JSONL data with updated descriptors saved to /home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.jsonl


In [14]:
import pandas as pd
import re
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

DEFAULT_NAMESPACE = "uri://ed-fi.org"

# Load descriptors CSV once and build a lookup table.
# The CSV file is assumed to have these columns:
# DESCRIPTOR_NAME,Owner,NAMESPACE,CODE_VALUE,SHORT_DESCRIPTION,DESCRIPTION
def load_descriptor_lookup(csv_path):
    df = pd.read_csv(csv_path)
    lookup = {}
    # Build a key: (descriptor_name, code_value_lower) -> "NAMESPACE#CODE_VALUE"
    for _, row in df.iterrows():
        descriptor_name = str(row['DESCRIPTOR_NAME']).strip()
        code_val = str(row['CODE_VALUE']).strip()
        namespace = str(row['NAMESPACE']).strip()
        lookup_key = (descriptor_name, code_val.lower())
        lookup[lookup_key] = f"{namespace}#{code_val}"
    return lookup

# Convert a JSON descriptor field name to the CSV descriptor name format.
def normalize_descriptor_field(json_field):
    # Insert underscores before uppercase letters (except first char) and convert to lower case.
    normalized = re.sub(r'(?<!^)(?=[A-Z])', '_', json_field).lower()
    if not normalized.endswith("s"):
        normalized += "s"
    return normalized

# Cache the lookup table from the Descriptors.csv file.
DESCRIPTOR_LOOKUP = load_descriptor_lookup('/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/Descriptors.csv')

def update_row_descriptors(row):
    """
    For each column that looks like it contains a descriptor (i.e. its header contains "Descriptor"),
    use the last token of the header (delimited by dot) for the lookup.
    If the descriptor value is found in the lookup, update the cell with its value;
    otherwise, log a warning and mark the row.
    Returns the updated row and a flag (True if any descriptor did not match).
    """
    no_match = False
    for col in row.index:
        if "Descriptor" in col:
            if pd.notna(row[col]):
                # Get last token after splitting by dot.
                last_token = col.split('.')[-1]
                # Remove any trailing bracket info
                last_token = re.sub(r'\[.*\]$', '', last_token)
                normalized_field = normalize_descriptor_field(last_token)
                lookup_key = (normalized_field, str(row[col]).strip().lower())
                if lookup_key in DESCRIPTOR_LOOKUP:
                    row[col] = DESCRIPTOR_LOOKUP[lookup_key]
                else:
                    no_match = True
                    logger.warning(f"No match for descriptor: header token '{last_token}' normalized as '{normalized_field}' with value '{row[col]}' (lookup key: {lookup_key})")
    return row, no_match

def process_csv(input_csv, output_csv, no_match_csv):
    df = pd.read_csv(input_csv)
    updated_rows = []
    no_match_rows = []
    
    for index, row in df.iterrows():
        row_updated, flag = update_row_descriptors(row.copy())
        updated_rows.append(row_updated)
        if flag:
            no_match_rows.append(row_updated)
    
    updated_df = pd.DataFrame(updated_rows)
    updated_df.to_csv(output_csv, index=False)
    logger.info(f"Updated CSV saved to {output_csv}")
    
    if no_match_rows:
        no_match_df = pd.DataFrame(no_match_rows)
        no_match_df.to_csv(no_match_csv, index=False)
        logger.info(f"Rows with no descriptor match saved to {no_match_csv}")
    else:
        logger.info("All rows had matching descriptor entries.")


In [9]:

input_csv = './data/schoolYearTypes/schoolYearTypes.csv'
output_csv = './output/schoolYearTypes/UpdatedSchoolYearTypes.csv'
no_match_csv = './output/schoolYearTypes/NoMatchSchoolYearTypes.csv'
process_csv(input_csv, output_csv, no_match_csv)

Updated CSV saved to ./output/schoolYearTypes/UpdatedSchoolYearTypes.csv
All rows had a matching descriptor.


In [ ]:

input_csv = './data/schoolYearTypes/schoolYearTypes.csv'
output_csv = './output/schoolYearTypes/UpdatedSchoolYearTypes.csv'
no_match_csv = './output/schoolYearTypes/NoMatchSchoolYearTypes.csv'
process_csv(input_csv, output_csv, no_match_csv)

In [16]:

input_csv = './data/stateEducationAgencies/stateEducationAgencies.csv'
output_csv = './output/stateEducationAgencies/UpdatedStateEducationAgencies.csv'
no_match_csv = './output/stateEducationAgencies/NoMatchStateEducationAgencies.csv'
process_csv(input_csv, output_csv, no_match_csv)

INFO: Updated CSV saved to ./output/stateEducationAgencies/UpdatedStateEducationAgencies.csv
INFO: All rows had matching descriptor entries.


In [4]:
import pandas as pd
import json
import re


def parse_path(path):
    """
    Parse a dot-delimited path string into components.
    Each component is a tuple of (name, index) where index is an integer if the component is an array element.
    For example: "addresses[0].periods[1].beginDate" becomes:
      [("addresses", 0), ("periods", 1), ("beginDate", None)]
    """
    components = []
    for part in path.split('.'):
        match = re.match(r'([^\[]+)(?:\[(\d+)\])?', part)
        if match:
            name, index = match.groups()
            components.append((name, int(index) if index is not None else None))
    return components

def recursive_set(obj, comps, value):
    """
    Recursively set the 'value' in the nested structure 'obj' using the list of components.
    Each component is a tuple (key, index). If index is provided, the key represents a list.
    """
    if not comps:
        return

    key, index = comps[0]

    # Final component: set the value
    if len(comps) == 1:
        if index is not None:
            if key not in obj:
                obj[key] = []
            while len(obj[key]) <= index:
                obj[key].append({})
            obj[key][index] = value
        else:
            obj[key] = value
        return

    # Not final: ensure the key exists and is of correct type (dict or list)
    if index is not None:
        if key not in obj:
            obj[key] = []
        while len(obj[key]) <= index:
            obj[key].append({})
        recursive_set(obj[key][index], comps[1:], value)
    else:
        if key not in obj:
            obj[key] = {}
        recursive_set(obj[key], comps[1:], value)

def set_nested_value(obj, components, value):
    recursive_set(obj, components, value)


def map_row_to_json(row):
    """
    Map a single CSV row to a nested JSON object.
    The CSV header paths (after stripping '/ed-fi/schools/') define the structure.
    For example, a header like:
      /ed-fi/schools/addresses[0].periods[0].beginDate
    will produce a nested structure where 'addresses' is an array of objects,
    and each address object has a 'periods' array of objects.
    
    The "SchoolYear" column is ignored.
    """
    json_obj = {}
    for col in row.index:
        if col == "SchoolYear":  # ignore the school year column
            continue
        if pd.notna(row[col]):
            path = re.sub(r'^/[^/]+/[^/]+/', '', col)
            components = parse_path(path)
            set_nested_value(json_obj, components, row[col])
    # Update descriptors in the resulting JSON object
    return json_obj


csv_file_path = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/UpdatedSchoolData.csv'
df = pd.read_csv(csv_file_path)
json_data = df.apply(map_row_to_json, axis=1).tolist()


jsonl_file_path = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.jsonl'
with open(jsonl_file_path, 'w') as f:
    for row_obj in json_data:
        f.write(json.dumps(row_obj) + "\n")

print(f"JSONL data with updated descriptors (ignoring SchoolYear) saved to {jsonl_file_path}")

JSONL data with updated descriptors (ignoring SchoolYear) saved to /home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/schools/data/SchoolData.jsonl
